Step 1: Load and prepare documents

## Document loaders in LangChain (basic overview)

Document loaders ingest content from various sources and convert it into LangChain `Document` objects. Each `Document` typically includes `page_content` (the text) and optional `metadata` (source, page number, URL, etc.).

### Common loader types
- **WebBaseLoader**: Load and parse web pages from URLs.
- **TextLoader**: Load plain text files.
- **PDFLoader** variants: Extract text from PDFs.
- **CSVLoader**: Load CSV rows into documents.
- **DirectoryLoader**: Batch-load files from a folder using a file pattern.

### Typical workflow
1. Initialize a loader with a source (file path, URL, or directory).
2. Call `.load()` to get a list of `Document` objects.
3. Optionally split documents into chunks before indexing or retrieval.

### Example (conceptual)
- Initialize a loader with a URL or file
- Call `.load()` to retrieve documents
- Pass the documents into a text splitter and vector store

Use document loaders to standardize data ingestion across different sources before downstream tasks like chunking, embedding, and retrieval.

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# List of URLs to load documents from
urls = ["https://lilianweng.github.io/posts/2023-06-23-agent/"]

# Load documents from the URLs
docs = [WebBaseLoader(
                url, 
                requests_kwargs={"headers": {"User-Agent": os.getenv("USER_AGENT")}}
            ).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
print(f"USER_AGENT: {os.getenv('USER_AGENT')}")

USER_AGENT: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36


Step 2: Split documents into chunks

In [3]:
# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

Step 3: Create a vector store

In [4]:
import os
from langchain_community.vectorstores import SKLearnVectorStore  
from langchain_openai import AzureOpenAIEmbeddings
from azure_openai_llm import create_azure_embedding

azure_embeddings = create_azure_embedding()

vectorstore = SKLearnVectorStore.from_documents(
            documents=doc_splits,
            embedding=azure_embeddings,
        )
      
retriever = vectorstore.as_retriever(k=4)
print(f"\n✅ Vector store created successfully!")


✅ Vector store created successfully!


Step 4: Set up the LLM and prompt template

In [5]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence
from azure_openai_llm import create_azure_llm

# Define the prompt template for the LLM
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,
    input_variables=["question", "documents"],
)

llm = create_azure_llm(type="chat", api="azure")

# Create a chain combining the prompt template and LLM
rag_chain = prompt | llm | StrOutputParser()

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


## `RunnableSequence` in LangChain

`RunnableSequence` is a LangChain class for chaining multiple runnable steps into one pipeline, where each step’s output becomes the next step’s input.

### What it does
- Composes stages like: **prompt → model → parser**
- Provides a single interface to run the full flow (`invoke`, `batch`, `stream`)
- Helps build clean, reusable LLM workflows (including RAG pipelines)

### In this notebook
Your chain:

- `prompt`
- `llm`
- `StrOutputParser()`

is a `RunnableSequence`, created either with pipe syntax:

```python
rag_chain = prompt | llm | StrOutputParser()
```

or explicitly:

```python
rag_chain = RunnableSequence(prompt, llm, StrOutputParser())
```

Both are equivalent.

### Why it is useful
- Keeps pipeline logic modular
- Makes debugging and swapping components easier
- Works naturally with retrievers and other runnables in LCEL

In [6]:
# Equivalent explicit forms: 
rag_chain = RunnableSequence(prompt, llm, StrOutputParser())

### Step 5: Invoke the chain

In [8]:
question_a = "What are the key components of an AI agent?"
question_b = "what is the weather in lahore"

question = question_a

# Retrieve relevant documents
documents = retriever.invoke(question)

# Extract content from retrieved documents
doc_texts = "\\n".join([doc.page_content for doc in documents])

# Get the answer from the language model
answer = rag_chain.invoke({"question": question, "documents": doc_texts})

print(f"\nQuestion: {question}\n")
print(f"Answer: {answer}")


Question: What are the key components of an AI agent?

Answer: The key components of an AI agent are planning (including task decomposition and self-reflection), memory (short-term and long-term), and tool use (calling external APIs for additional information or capabilities). The LLM acts as the agent’s brain, orchestrating these components to solve complex tasks efficiently.


Step 5: Integrate the retriever and LLM into a RAG application

In [9]:
# Define a class to encapsulate the retriever and RAG chain

class RAGApplication:
    def __init__(self, retriever, rag_chain):
        self.retriever = retriever
        self.rag_chain = rag_chain
        
    def run(self, question):
        # Retrieve relevant documents
        documents = self.retriever.invoke(question)
        
        # Extract content from retrieved documents
        doc_texts = "\\n".join([doc.page_content for doc in documents])
        
        # Get the answer from the language model
        answer = self.rag_chain.invoke({"question": question, "documents": doc_texts})
        return answer


Step 6: Test your RAG application

In [10]:
# Create and test the RAG application
if 'vectorstore' in locals() and vectorstore is not None and 'rag_chain' in locals():
    rag_app = RAGApplication(retriever, rag_chain)

    # Test with some questions
    questions = [
        "What are the key components of an AI agent?",
        "How do adversarial attacks work on large language models?"
    ]

    # Test each question
    for question in questions:
        print(f"\n🤖 Question: {question}")
        print("-" * 50)
        try:
            answer = rag_app.run(question)
            print(f"📋 Answer: {answer}")
        except Exception as e:
            print(f"❌ Error: {e}")
        print("=" * 50)
else:
    print("🚨 Cannot test RAG application - setup incomplete.")
    print("Please run the previous cells to fix configuration issues.")


🤖 Question: What are the key components of an AI agent?
--------------------------------------------------
📋 Answer: The key components of an AI agent are Planning (task decomposition and self-reflection), Memory (short-term and long-term memory for information retention and recall), and Tool Use (calling external APIs for additional information and capabilities). The LLM acts as the agent's brain, coordinating these components for problem-solving.

🤖 Question: How do adversarial attacks work on large language models?
--------------------------------------------------
📋 Answer: The provided documents do not contain information on how adversarial attacks work on large language models. Therefore, I don't know the answer based on these sources.


### `RunnablePassthrough` and `RunnableLambda` (quick intro)

- **`RunnablePassthrough`**: Forwards input unchanged.  
    Useful when a value (like the user question) should move through a chain while other fields are computed in parallel.

- **`RunnableLambda`**: Wraps a Python function as a runnable step.  
    Useful for lightweight transformations, such as formatting retrieved documents before prompting.

```python
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

chain = {
        "question": RunnablePassthrough(),              # keep original input
        "context": retriever | RunnableLambda(format_docs),  # transform docs -> text
}
```

In short: **Passthrough keeps data as-is; Lambda applies custom logic.**

In [11]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Better RAG chain: retrieval + formatting + generation in one LCEL pipeline
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

better_prompt = PromptTemplate(
    template="""You are a precise QA assistant.
Use only the context below to answer the question.
If the answer is not in the context, say "I don't know."
Keep the response to at most 3 sentences.

Question: {question}
Context:
{context}

Answer:""",
    input_variables=["question", "context"],
)

# In this LCEL pipeline, the input is the user question (a plain string).
# - "question": RunnablePassthrough() forwards that original question unchanged.
# - "context": retriever gets relevant docs for the same question, then format_docs
#   converts those documents into one context string for the prompt.
# The prompt then receives both fields: {question, context}.
better_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
    }
    | better_prompt
    | llm
    | StrOutputParser()
)

print(better_rag_chain.invoke("What are the key components of an AI agent?"))

The key components of an AI agent are Planning (including subgoal decomposition and self-reflection), Memory (short-term and long-term memory), and Tool use (calling external APIs for additional information or capabilities). These components work together with the LLM as the core controller or brain of the agent.
